# Практическая работа № 7
## Архитектура игры: ECS, MVC, слои, async/threading

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр
**Модуль:** 7 — Архитектура игры
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно
**Формат сдачи:** Git-репозиторий + тесты + README + ADR
**Максимальный балл:** 5

---

## О чём эта работа

После модуля 6 проект умеет публиковать события, создавать объекты через фабрики и управлять ресурсами. Но:
- сущности всё ещё классы-«толстяки»;
- код лежит в одной куче;
- загрузка ассетов блокирует игру.

В этой работе вы:
- спроектируете ECS;
- разложите проект по слоям;
- добавите асинхронную загрузку и фоновый поток.

## Цели работы

После выполнения вы сможете:
- проектировать ECS (Entity, Component, System);
- разделять данные и логику;
- раскладывать код по слоям (domain, application, infrastructure, presentation);
- применять MVC/MVVM;
- использовать threading и asyncio для I/O;
- описывать риски многопоточности.

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | ветка Git |
| 1 | Проектирование ECS | компоненты и системы |
| 2 | Реализация ECS | Entity, Component, System |
| 3 | Разделение на слои | domain, application, ... |
| 4 | Асинхронная загрузка ассетов | asyncio |
| 5 | Поток для фоновой задачи | threading, Queue |
| 6 | Рефакторинг под слои | перенос кода |
| 7 | Юнит-тесты | тесты систем и слоёв |
| 8 | Документация | ADR-007, схема слоёв |

**Важно:** этот блокнот содержит только задания. Готовые решения — в `module7_example.ipynb` (образец, не для копирования).

**Для углублённого изучения теории:**
- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html)
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq)
- Документация Python: [threading](https://docs.python.org/3/library/threading.html), [asyncio](https://docs.python.org/3/library/asyncio.html)

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Подготовка

```bash
git checkout main
git checkout -b module7-ecs-layers
pytest -v
```

Все тесты из модулей 1–6 должны проходить.

---
# Шаг 1. Проектирование ECS

## Задание 1.1. Список компонентов

Опишите компоненты для вашей игры. Минимум:
- `Position` — координаты;
- `Velocity` — скорость;
- `Health` — здоровье.

Опционально: `Drawable`, `Inventory`, `Collider`.

Для каждого компонента укажите поля и типы.

**Сохраните в** `docs/ecs_module7.md`.

## Задание 1.2. Список систем

Опишите системы. Минимум:
- `MovementSystem` — двигает сущности с `Position` + `Velocity`;
- `HealthSystem` — следит за `Health`.

Укажите, какие компоненты читает/пишет каждая система.

## Задание 1.3. Ответьте

1. Какие сущности из проекта станут просто наборами компонентов?
2. Какие системы заменят методы классов?
3. Что останется классами (не ECS)?

---
# Шаг 2. Реализация ECS

## Задание 2.1

Создайте `domain/ecs.py`:
1. `Entity` — идентификатор + dict компонентов.
2. Компоненты как `@dataclass` (`Position`, `Velocity`, `Health`).
3. `MovementSystem` — обновляет позиции.
4. `HealthSystem` — обрабатывает здоровье.

**Требования:**
- Системы не знают про конкретные классы сущностей.
- Системы обрабатывают только сущности с нужными компонентами.
- `dt` передаётся в `update`.

## Задание 2.2. Коммит

```bash
git add domain/ecs.py
git commit -m "Add lightweight ECS: Entity, components, systems"
```

---
# Шаг 3. Разделение на слои

## Задание 3.1. Создайте структуру слоёв

```bash
mkdir -p domain application infrastructure presentation
```

Распределите модули:

| Слой | Модули |
|---|---|
| `domain/` | `ecs.py`, `world_state.py`, компоненты |
| `application/` | `game.py`, `game_loop.py`, системы |
| `infrastructure/` | `audio_manager.py`, `config.py`, загрузчики |
| `presentation/` | `renderer.py`, `ui.py`, `input_handler.py` |

**Требования:**
- domain не импортирует application/infrastructure/presentation.
- presentation импортирует application и domain.
- infrastructure реализует интерфейсы domain/application.

## Задание 3.2. Ответьте

1. Какие модули из старых `core/`, `game/`, `entities/` куда переедут?
2. Есть ли циклические зависимости? Как их разорвать?

## Задание 3.3. Коммит

```bash
git add .
git commit -m "Reorganize project into layers"
```

---
# Шаг 4. Асинхронная загрузка ассетов

## Задание 4.1

Создайте `infrastructure/asset_loader.py`:

```python
async def load_assets(paths):
    return await asyncio.gather(*(load_one(p) for p in paths))
```

**Требования:**
- Имитация I/O через `asyncio.sleep`.
- Параллельная загрузка нескольких ассетов через `gather`.
- Результат — словарь `path -> data`.

## Задание 4.2. Коммит

```bash
git add infrastructure/asset_loader.py
git commit -m "Add async asset loader"
```

---
# Шаг 5. Поток для фоновой задачи

## Задание 5.1

Создайте `infrastructure/background.py`:
1. Фоновый поток с `threading.Thread(daemon=True)`.
2. `queue.Queue` для передачи сообщений.
3. `threading.Event` для остановки.

**Требования:**
- Поток читает сообщения из очереди и выводит/логирует их.
- Корректное завершение через `Event`.
- Данные передаются только через очередь.

## Задание 5.2. Коммит

```bash
git add infrastructure/background.py
git commit -m "Add background thread with Queue"
```

---
# Шаг 6. Рефакторинг под слоистую архитектуру

## Задание 6.1

Перенесите весь код проекта в новую структуру слоёв. Убедитесь, что:
- `Game` (application) собирает всё через конструкторы;
- domain не знает про PyGame;
- загрузка ассетов вызывается из application;
- presentation подписывается на события через EventBus.

## Задание 6.2. Проверка

Запустите игру. Убедитесь, что:
- игра работает как раньше;
- ассеты загружаются асинхронно;
- фоновый поток логирует сообщения;
- цикл не блокируется.

## Задание 6.3. Коммит

```bash
git add .
git commit -m "Refactor project to layered architecture"
```

---
# Шаг 7. Юнит-тесты

## Задание 7.1. Тесты ECS
`tests/test_ecs.py`:
- `test_entity_add_component`;
- `test_movement_system_moves`;
- `test_movement_ignores_without_velocity`;
- `test_health_system`.

## Задание 7.2. Тесты загрузчика
`tests/test_asset_loader.py`:
- `test_load_assets_parallel`;
- `test_result_is_dict`.

## Задание 7.3. Тесты фонового потока
`tests/test_background.py`:
- `test_queue_delivers_messages`;
- `test_stop_event_terminates`.

## Задание 7.4. Запуск

```bash
pytest -v
```

Все тесты должны проходить.

---
# Шаг 8. Документация

## Задание 8.1. README
Обновите разделы:
- архитектура (ECS, слои);
- структура папок;
- описание async/threading-решений.

## Задание 8.2. ADR-007
`docs/adr_007_architecture.md`:
- Контекст: монолитный код, блокирующий I/O.
- Решение: ECS + слои + async/threading.
- Альтернативы: оставить ООП-иерархию, синхронная загрузка.
- Последствия: плюсы, минусы, риски потоков.

## Задание 8.3. Схема слоёв
Добавьте в `docs/architecture_module7.md` схему слоёв и направление зависимостей.

## Задание 8.4. PEP 8

```bash
pycodestyle domain/ application/ infrastructure/ presentation/ main.py
```

Исправьте замечания.

## Задание 8.5. Финальный коммит

```bash
git add .
git commit -m "Document module 7: ECS, layers, async/threading"
git checkout main
git merge module7-ecs-layers
```

---
# Чек-лист
```
[ ] ECS спроектирован
[ ] Entity, компоненты, системы реализованы
[ ] Данные и логика разделены
[ ] Слои domain/application/infrastructure/presentation созданы
[ ] Зависимости идут внутрь
[ ] Асинхронная загрузка ассетов работает
[ ] Фоновый поток работает и завершается
[ ] Игра не блокируется при загрузке
[ ] Все тесты проходят
[ ] README, ADR, схема слоёв обновлены
[ ] PEP 8 без замечаний
```

---

# Критерии оценки

| Критерий | Баллы |
|---|---:|
| ECS | 1.5 |
| Слоистая архитектура | 1.5 |
| async/threading | 1.0 |
| Интеграция | 0.5 |
| Тесты | 0.5 |
| **Итого** | **5.0** |

**Штрафы:** −1 за задержку, −2 за плагиат, −1 за отсутствие тестов.

**Бонусы (+1):** archetype ECS, несколько фоновых потоков, сетевая заглушка на asyncio.

## Что дальше
В модуле 8 — тестирование, рефакторинг и подготовка к защите.